Next steps: 
- test on variety of adatas
- make sure it works with csvs too 



In [1]:
import numpy as np
from matplotlib import pyplot as plt


from scphere.model.vae import SCPHERE

import pandas as pd
import anndata as ad

import plotly.graph_objects as go
import plotly.express as px

from scipy.sparse import issparse
from scphere.util.trainer import Trainer

import sys
sys.path.append('/projects/steiflab/research/hmacdonald/total_RNA/Ouroboros_pypi/ouroboros')

from ouroboros_functions import *


/projects/steiflab/research/hmacdonald/applications/python/miniconda3/envs/old_sphere3.6/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:516: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
/projects/steiflab/research/hmacdonald/applications/python/miniconda3/envs/old_sphere3.6/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:517: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
/projects/steiflab/research/hmacdonald/applications/python/miniconda3/envs/old_sphere3.6/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:518: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future versi

In [2]:
 
def run_ouroboros(data, data_type, species = 'human', outdir = '.'):
    """
    Run the full Ouroboros pipeline for projecting single-cell expression data
    into VAE spherical embedding space and using KNN to compute cell cycle phase, pseudotime and dormancy pseudotime.

    Parameters
    ----------
    data : str 
        Path to input file (needs to be either a 'h5ad' or a 'csv')
        - For 'h5ad': Anndata object - Ouroboros expects raw counts in .layers['raw_counts']
        - For 'csv' : expects a CSV with cells as rows and genes as columns, with a 'cell_id' index column - must be RAW COUNTS

    data_type : str
        Format of the input data, must be either 'h5ad' or 'csv'

    species : str {'human', 'mouse'}, optional (default='human')
        Species of origin. If 'mouse', gene names will be mapped to human orthologs.

    outdir : str, optional (default='.')
        Output directory where embedding and pseudotime files will be saved.

    Returns
    -------
    z_df : pandas.DataFrame
        A dataframe containing the spherical embedding coordinates (dim1, dim2, dim3),
        predicted states, and dormancy pseudotime for each cell.

    Outputs
    -------
    - ouroboros_embeddings_pseudotimes.csv
    - retrained_reference_embeddings.csv (if retraining is triggered)

    Notes
    -----
    - If key training genes are missing from the input data, the model will be retrained
      on the subset of genes available.
    - The output embeddings and pseudotimes can be used for visualization and downstream analysis.

    Example
    -------
    >>> run_ouroboros("sample.h5ad", "h5ad", species="human", outdir="results/")
    >>> python -m Ouroboros_pypi.Ouroboros.main \
    --data adata.h5ad  \
    --data_type h5ad \
    --species mouse \
    --outdir /path/to/outdir
    """

    if data_type == 'h5ad':
        data = ad.read_h5ad(data)
        data.X = data.layers['raw_counts'].copy()
        
    elif data_type == 'csv':
        data = pd.read_csv(data)
        data = data.set_index('cell_id')
    else: 
        raise TypeError("Unsupported data type. Expected --h5ad or --csv for data_type.")


    if species == 'mouse':
        data = convert_to_human_genes(data)
        if isinstance(data, ad.AnnData):
            data.layers['raw_counts'] = data.X.copy()
    elif species == 'human':
        pass
    else:
        raise TypeError("Unsupported species. Model only optimized for --human or --mouse")

    missing = check_features(data)

    if len(missing) > 0:
 
        model, ref_embed, in_order_feature_set = ouroboros_retrain(data)
        ref_embed.to_csv(f'{outdir}/retrained_reference_embeddings.csv')
        z_df = embed_in_retrained_sphere(data, model, in_order_feature_set)
        z_df = KNN_predict(ref_embed, z_df)
    
        cc_df = calculate_cell_cycle_pseudotime(z_df, ref_embed,  phase_category = 'KNN_phase')
        cc_df = cc_df[['cell_cycle_pseudotime']]
        z_df = z_df.merge(cc_df, how = 'left', left_index = True, right_index = True)
        pseud, ref_pseud = dormancy_depth(z_df, ref_embed, retrained = True)
        z_df = z_df.merge(pseud, how = 'left', left_index = True, right_index = True)
        z_df.to_csv(f'{outdir}/ouroboros_embeddings_pseudotimes.csv')
        plot_sphere(z_df, colour_by = 'cell_cycle_pseudotime', palette = None, ref = ref_embed, velocity = None, marker_size = 2, cycle_pole = reference_CC_pole_point, savefig = f'{outdir}/ouroboros_cell_cycle_pseudotime.html', show = False)
        plot_sphere(z_df, colour_by = 'dormancy_depth', palette = None, ref = ref_embed, velocity = None, marker_size = 2, cycle_pole = reference_CC_pole_point, savefig = f'{outdir}/ouroboros_dormancy_depth.html', show = False)
        return pseud, z_df
        
    else:
        matrix = ouroboros_preprocess(data, data_type, species = 'human')
        z_df = ouroboros_embed(matrix, data, data_type, outdir = outdir)
        # Read in known reference embeddings 
        ref_embed = pd.read_csv(DATA_DIR / 'reference_embeddings.csv')
        # set cell id to be index
        ref_embed = ref_embed.set_index('cell_id')
        plot_sphere(z_df, colour_by = 'cell_cycle_pseudotime', palette = None, ref = ref_embed, velocity = None, marker_size = 2, cycle_pole = reference_CC_pole_point, savefig = f'{outdir}/ouroboros_cell_cycle_pseudotime.html', show = False)
        plot_sphere(z_df, colour_by = 'dormancy_depth', palette = None, ref = ref_embed, velocity = None, marker_size = 2, cycle_pole = reference_CC_pole_point, savefig = f'{outdir}/ouroboros_dormancy_depth.html', show = False)
        z_df.to_csv(f'{outdir}/ouroboros_embeddings_pseudotimes.csv')
        return z_df

In [3]:
data = '/projects/steiflab/research/hmacdonald/total_RNA/data/ovarian/ovarian_scratch/longitudinal/processed/ovarian_preprocessed.h5ad'
data_type = 'h5ad'

In [4]:
pseud, z_df = run_ouroboros(data, data_type, species = 'human', outdir = '.')


The TensorFlow contrib module will not be included in TensorFlow 2.0.
For more information, please see:
  * https://github.com/tensorflow/community/blob/master/rfcs/20180907-contrib-sunset.md
  * https://github.com/tensorflow/addons
  * https://github.com/tensorflow/io (for I/O related ops)
If you depend on functionality not listed there, please file an issue.



2025-05-21 13:39:19,803: WARNING: 
The TensorFlow contrib module will not be included in TensorFlow 2.0.
For more information, please see:
  * https://github.com/tensorflow/community/blob/master/rfcs/20180907-contrib-sunset.md
  * https://github.com/tensorflow/addons
  * https://github.com/tensorflow/io (for I/O related ops)
If you depend on functionality not listed there, please file an issue.



Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor


2025-05-21 13:39:19,827: WARNING: From /projects/steiflab/research/hmacdonald/applications/python/miniconda3/envs/old_sphere3.6/lib/python3.6/site-packages/tensorflow/python/ops/init_ops.py:1251: calling VarianceScaling.__init__ (from tensorflow.python.ops.init_ops) with dtype is deprecated and will be removed in a future version.
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor


Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where


2025-05-21 13:39:20,282: WARNING: From /projects/steiflab/research/hmacdonald/applications/python/miniconda3/envs/old_sphere3.6/lib/python3.6/site-packages/tensorflow/python/ops/math_ops.py:2403: add_dispatch_support.<locals>.wrapper (from tensorflow.python.ops.array_ops) is deprecated and will be removed in a future version.
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where


Instructions for updating:
tf.py_func is deprecated in TF V2. Instead, there are two
    options available in V2.
    - tf.py_function takes a python function which manipulates tf eager
    tensors instead of numpy arrays. It's easy to convert a tf eager tensor to
    an ndarray (just call tensor.numpy()) but having access to eager tensors
    means `tf.py_function`s can use accelerators such as GPUs as well as
    being differentiable using a gradient tape.
    - tf.numpy_function maintains the semantics of the deprecated tf.py_func
    (it is not differentiable, and manipulates numpy arrays). It drops the
    stateful argument making all functions stateful.
    


2025-05-21 13:39:21,416: WARNING: From /projects/steiflab/research/hmacdonald/applications/python/miniconda3/envs/old_sphere3.6/lib/python3.6/site-packages/scPhere-0.1.0-py3.6.egg/scphere/ops/ive.py:23: py_func (from tensorflow.python.ops.script_ops) is deprecated and will be removed in a future version.
Instructions for updating:
tf.py_func is deprecated in TF V2. Instead, there are two
    options available in V2.
    - tf.py_function takes a python function which manipulates tf eager
    tensors instead of numpy arrays. It's easy to convert a tf eager tensor to
    an ndarray (just call tensor.numpy()) but having access to eager tensors
    means `tf.py_function`s can use accelerators such as GPUs as well as
    being differentiable using a gradient tape.
    - tf.numpy_function maintains the semantics of the deprecated tf.py_func
    (it is not differentiable, and manipulates numpy arrays). It drops the
    stateful argument making all functions stateful.
    


0 / 11000 {'Log-likelihood': -415.44635, 'ELBO': -415.7945, 'KL': 0.34815645}
50 / 11000 {'Log-likelihood': -300.4795, 'ELBO': -301.2874, 'KL': 0.80793023}
100 / 11000 {'Log-likelihood': -314.35883, 'ELBO': -315.44376, 'KL': 1.0849357}
150 / 11000 {'Log-likelihood': -292.41748, 'ELBO': -294.20547, 'KL': 1.7880039}
200 / 11000 {'Log-likelihood': -281.9276, 'ELBO': -284.0017, 'KL': 2.074099}
250 / 11000 {'Log-likelihood': -251.9809, 'ELBO': -254.35088, 'KL': 2.3699794}
300 / 11000 {'Log-likelihood': -266.70758, 'ELBO': -269.3145, 'KL': 2.606924}
350 / 11000 {'Log-likelihood': -256.8778, 'ELBO': -259.68152, 'KL': 2.803699}
400 / 11000 {'Log-likelihood': -261.43195, 'ELBO': -264.39923, 'KL': 2.9672823}
450 / 11000 {'Log-likelihood': -270.0504, 'ELBO': -273.12827, 'KL': 3.0778556}
500 / 11000 {'Log-likelihood': -257.2107, 'ELBO': -260.38083, 'KL': 3.170144}
550 / 11000 {'Log-likelihood': -250.75633, 'ELBO': -254.01282, 'KL': 3.2564902}
600 / 11000 {'Log-likelihood': -256.9702, 'ELBO': -260.

In [11]:
pseud

,dormancy_depth
AAACCTGCAGGTTTCA-EOC372_pPer,-0.422304
AAACCTGGTCCGAATT-EOC372_pPer,-0.460572
AAAGATGCATCTGGTA-EOC372_pPer,-0.373832
AAAGTAGTCGCTTAGA-EOC372_pPer,-0.549943
AAATGCCAGGTGCACA-EOC372_pPer,-0.268145
...,...
TTTGCGCCACATCCAA-EOC443_pOme,-0.264016
TTTGCGCCACGTCAGC-EOC443_pOme,-0.812128
TTTGCGCCATTCACTT-EOC443_pOme,-0.428861
TTTGTCACATTGGGCC-EOC443_pOme,-0.309478
